# Visualisation Generation for MSc Dissertation

## 1. Imoprts and Global Setup

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, LogLocator

sns.set_theme(style="whitegrid", font_scale=1.15)
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

os.makedirs("../figures/appendix", exist_ok=True)

## 2. Configuration

In [ ]:
DATASETS = ["FD001", "FD002"]
MODELS = {
    "Baseline LSTM": "lstm_baseline",
    "Stacked LSTM": "lstm_improved",
    "1D-CNN": "cnn",
    "Hybrid CNN-LSTM": "cnn_lstm"
}
METRICS = ["rmse", "mae", "nasa"]
METRIC_LABELS = {
    "rmse": "RMSE (cycles)",
    "mae": "MAE (cycles)",
    "nasa": "NASA Score"
}

# Readable formatter for large NASA Scores
def nasa_formatter(x, pos):
    if x >= 1_000_000:
        return f'{x/1_000_000:.1f}M'
    elif x >= 1_000:
        return f'{x/1_000:.0f}k'
    else:
        return f'{x:.0f}'

nasa_format = FuncFormatter(nasa_formatter)

## 3. Load in Seed-level Metrics

In [ ]:
records = []
for ds in DATASETS:
    for name, prefix in MODELS.items():
        path = f"../results/metrics/{prefix}_{ds}_metrics.csv"
        if not os.path.exists(path):
            print(f"WARNING: Missing {path}")
            continue
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            for m in METRICS:
                records.append({
                    "Dataset": ds,
                    "Model": name,
                    "Metric": m,
                    "Value": row[m],
                    "Seed": row.get("seed", None)
                })

long_df = pd.DataFrame(records)
print(f"Loaded {len(long_df)} metric records")
print(long_df.groupby(["Dataset", "Model", "Metric"]).size().unstack(fill_value=0))

## 4. Seed-level Box Plots

In [ ]:
print("\nGenerating seed-level boxplots...")

for ds in DATASETS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, metric in zip(axes, METRICS):
        subset = long_df[(long_df["Dataset"] == ds) & (long_df["Metric"] == metric)]
        sns.boxplot(
            data=subset,
            x="Model",
            y="Value",
            ax=ax,
            palette="muted",
            width=0.6,
            fliersize=3,
            linewidth=1.2
        )
        ax.set_title(f"{ds} – {METRIC_LABELS[metric]}", fontweight="bold")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=18)

        if metric == "nasa":
    
            if ds == "FD001":
                ax.set_ylim(6000, 16000)
                ticks = [6000, 10000, 14000]
            else:
                ax.set_ylim(25000, 200000)
                ticks = [25000, 50000, 75000, 100000, 125000, 150000, 175000, 200000]
    
            ax.set_yticks(ticks)
            ax.yaxis.set_major_formatter(nasa_format)
            ax.yaxis.set_minor_formatter(plt.NullFormatter())

    plt.suptitle(f"Seed-level Performance Distributions – {ds} (10 seeds)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(f"../figures/appendix/boxplots_{ds.lower()}.png")
    plt.show()

## 5. Grouped Bar Graphs with Error Bars

In [ ]:
print("\nGenerating grouped bar plots...")

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, metric in zip(axes, METRICS):
    subset = long_df[long_df["Metric"] == metric]
    sns.barplot(
        data=subset,
        x="Dataset",
        y="Value",
        hue="Model",
        ax=ax,
        errorbar="sd",
        capsize=0.12,
        err_kws={"linewidth": 1.3},
        palette="muted",
        edgecolor="black",
        linewidth=0.7
    )
    ax.set_title(METRIC_LABELS[metric], fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(METRIC_LABELS[metric])
    
    if metric == "nasa":
        ax.set_yscale("log")
        ticks = [5000, 10000, 20000, 50000, 100000, 200000]
        ax.set_yticks(ticks)
        ax.yaxis.set_major_formatter(nasa_format)
    
        ax.yaxis.set_minor_formatter(plt.NullFormatter())

    ax.legend_.remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4,
           bbox_to_anchor=(0.5, 1.08), frameon=True)
plt.suptitle("Mean Performance ± 1 Standard Deviation (10 seeds)", fontsize=14, y=1.12)
plt.tight_layout()
plt.savefig("../figures/appendix/grouped_bar_metrics.png")
plt.show()

## 6. Training and Learning Curves

In [ ]:
print("\nGenerating training curves (if histories are available)...")

def plot_training_curve(history_path, title, save_name):
    if not os.path.exists(history_path):
        print(f"  Skipping (not found): {history_path}")
        return False

    hist = pd.read_csv(history_path)
    epoch_col = "epoch" if "epoch" in hist.columns else hist.columns[0]
    train_col = next((c for c in hist.columns if "train" in c.lower() and "rmse" in c.lower()), None)
    val_col   = next((c for c in hist.columns if "val" in c.lower() and "rmse" in c.lower()), None)

    if train_col is None or val_col is None:
        print(f"  Could not find train/val RMSE columns in {history_path}")
        return False

    plt.figure(figsize=(8, 5))
    plt.plot(hist[epoch_col], hist[train_col], label="Train RMSE", linewidth=2)
    plt.plot(hist[epoch_col], hist[val_col],   label="Validation RMSE", linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("RMSE (cycles)")
    plt.title(title, fontweight="bold")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"../figures/appendix/{save_name}")
    plt.show()
    return True

history_candidates = [
    # Baseline LSTM
    ("../results/histories/lstm_baseline_FD001.csv", "Baseline LSTM – FD001", "train_curve_lstm_baseline_fd001.png"),
    ("../results/histories/lstm_baseline_FD002.csv", "Baseline LSTM – FD002", "train_curve_lstm_baseline_fd002.png"),
    
    # Stacked LSTM (previously called Improved LSTM)
    ("../results/histories/lstm_improved_FD001.csv", "Stacked LSTM – FD001", "train_curve_lstm_fd001.png"),
    ("../results/histories/lstm_improved_FD002.csv", "Stacked LSTM – FD002", "train_curve_lstm_fd002.png"),
    
    # 1D-CNN
    ("../results/histories/cnn_FD001.csv",           "1D-CNN – FD001",        "train_curve_cnn_fd001.png"),
    ("../results/histories/cnn_FD002.csv",           "1D-CNN – FD002",        "train_curve_cnn_fd002.png"),
    
    # Hybrid CNN-LSTM
    ("../results/histories/cnn_lstm_FD001.csv",      "Hybrid CNN-LSTM – FD001","train_curve_hybrid_fd001.png"),
    ("../results/histories/cnn_lstm_FD002.csv",      "Hybrid CNN-LSTM – FD002","train_curve_hybrid_fd002.png"),
]

found_any = False
for path, title, save_name in history_candidates:
    if plot_training_curve(path, title, save_name):
        found_any = True

if not found_any:
    print("  No training history files found.")
    print("  You can still include the training curves you already generated manually.")

## 7. Predicted vs Actual Scatter plots

Using seed 42

In [ ]:
print("\nGenerating Predicted vs Actual scatter plots (seed 42)...")

import glob

def plot_scatter(y_true, y_pred, title, save_name):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.35, color="#3176DA", s=12)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val],
             linestyle="--", color="#C02F25", linewidth=2, label="Ideal Prediction")
    plt.xlabel("Actual RUL")
    plt.ylabel("Predicted RUL")
    plt.title(title, fontweight="bold")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"../figures/appendix/{save_name}")
    plt.show()
    print(f"  Saved: {save_name}")

# Preferred prefixes for each model
MODEL_INFO = {
    "Baseline LSTM":   {"prefix": "lstm_baseline", "file_stub": "lstm_baseline_pred"},
    "Stacked LSTM":    {"prefix": "lstm_improved", "file_stub": "lstm_improved_pred"},
    "1D-CNN":          {"prefix": "cnn",           "file_stub": "cnn_pred"},
    "Hybrid CNN-LSTM": {"prefix": "cnn_lstm",      "file_stub": "cnn_lstm_pred"},
}

for ds in DATASETS:
    y_true_path = f"../data/processed/y_val_{ds}_engine_split.npy"
    if not os.path.exists(y_true_path):
        print(f"  Ground truth not found for {ds}")
        continue
    y_true = np.load(y_true_path)

    for model_name, info in MODEL_INFO.items():
        seed42_path = f"../results/preds/{info['file_stub']}_{ds}_seed42.npy"
        
        if os.path.exists(seed42_path):
            y_pred = np.load(seed42_path).flatten()
        else:
            pattern = f"../results/preds/{info['file_stub']}_{ds}_seed*.npy"
            matches = sorted(glob.glob(pattern))
            if not matches:
                print(f"  Predictions not found for {model_name} – {ds}")
                continue
            seed42_path = matches[0]
            y_pred = np.load(seed42_path).flatten()
            print(f"  (Using {os.path.basename(seed42_path)} instead of seed 42)")

        n = min(len(y_true), len(y_pred))
        save_name = f"scatter_{info['prefix']}_{ds.lower()}.png"
        plot_scatter(
            y_true[:n],
            y_pred[:n],
            f"{model_name}: Predicted vs Actual RUL ({ds})",
            save_name
        )

print("\nAll available appendix figures have been saved to ../figures/appendix/")
print("\nAll available appendix figures have been saved to ../figures/appendix/")